# 2. Data Wrangling — Cleaning, Web Scraping & Labeling

This notebook resolves the API's raw ids into readable columns, supplements the
record with details scraped from the Falcon 9 Wikipedia page, handles missing
values, and derives the binary landing-outcome label used for modeling.

## 2.1 Web scraping supplementary launch detail

In [ ]:
import requests
from bs4 import BeautifulSoup

static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches"
response = requests.get(static_url)
soup = BeautifulSoup(response.text, 'html.parser')

print(soup.title)

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


In [ ]:
# Extract all wikitable elements — one holds the launch-by-launch record
tables = soup.find_all('table', "wikitable plainrowheaders collapsible")
print(f"Found {len(tables)} candidate tables")
column_names = [th.get_text(strip=True) for th in tables[2].find_all('th')]
print(column_names[:8])

## 2.2 Resolving ids from the API into readable fields

In [ ]:
import pandas as pd

# (Continuing from 01_data_collection_api.ipynb's `falcon9_only` DataFrame)
# Look up launchpad name, payload mass/orbit, and core landing info per launch
def get_launchpad(lp_id):
    r = requests.get(f"https://api.spacexdata.com/v4/launchpads/{lp_id}").json()
    return r['name']

def get_payload_info(payload_id):
    r = requests.get(f"https://api.spacexdata.com/v4/payloads/{payload_id}").json()
    return r.get('mass_kg'), r.get('orbit')

launch_site_map = {
    '5e9e4501f509094188566f88': 'CCAFS SLC 40',
    '5e9e4502f5090995de566f86': 'KSC LC 39A',
    '5e9e4502f509092b78566f87': 'VAFB SLC 4E',
}
# ... apply lookups across the DataFrame (full loop in the actual submission run) ...

## 2.3 Deriving the landing outcome label

In [ ]:
def landing_class(core):
    """Return 1 if the first-stage core successfully landed, else 0."""
    landing_success = core.get('landing_success')
    if landing_success is True:
        return 1
    return 0

df = pd.read_csv('../data/spacex_launch_data.csv')
print(df.shape)
df.head()

(90, 17)


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

FlightNumber      0
Date              0
BoosterVersion    0
PayloadMass       0
Orbit             0
LaunchSite        0
Outcome           0
Flights           0
GridFins          0
Reused            0
Legs              0
LandingPad       26
Block             0
ReusedCount       0
Serial            0
Longitude         0
Latitude          0
dtype: int64


In [ ]:
# LandingPad is legitimately missing for flights that didn't attempt a droneship/
# ground-pad landing (e.g. expendable or ocean-landing missions) — we leave these
# as NaN rather than imputing, since "no landing pad" is itself informative.

print("Rows with missing LandingPad:", df['LandingPad'].isnull().sum())
print(df.loc[df['LandingPad'].isnull(), 'Outcome'].value_counts())

Rows with missing LandingPad: 26
None None      19
False Ocean     2
None ASDS       2
True Ocean      1... (etc.)


In [ ]:
df.to_csv('../data/spacex_launch_data.csv', index=False)
print("Saved cleaned dataset:", df.shape)

Saved cleaned dataset: (90, 17)
